# Predicting Trip-Level Driver Pay

In [3]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import duckdb
con = duckdb.connect()

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
RANDOM_STATE = 42

## Variables

In [8]:
import os
os.chdir('/Users/gemie/orie3120projver2/')

In [20]:
zones = pd.read_csv("taxi_zone_lookup.csv")

all_vars = con.sql("""
    SELECT
        pickup_datetime,
        z.Borough                    AS borough,
        trip_time,
        driver_pay,
        HOUR(pickup_datetime)        AS hour,
        DAYOFWEEK(pickup_datetime)   AS dow,
        (CASE WHEN DOLocationID = 132 THEN 1 ELSE 0 END) AS is_JFK,
        (CASE WHEN DOLocationID = 138 THEN 1 ELSE 0 END) AS is_LGA,
        (CASE WHEN DOLocationID = 1   THEN 1 ELSE 0 END) AS is_EWR
    FROM read_parquet(['2024/*.parquet', '2025/*.parquet']) a
    JOIN zones z ON a.PULocationID = z.LocationID
    WHERE trip_time BETWEEN 120 AND 7200
      AND driver_pay > 5
      AND z.Borough IS NOT NULL
      AND z.Borough NOT IN ('Unknown', 'N/A')
    USING SAMPLE 2000000 ROWS
""").df()

all_vars["pickup_datetime"]   = pd.to_datetime(all_vars["pickup_datetime"])
all_vars["borough"]           = all_vars["borough"].astype("category")
all_vars["trip_duration_min"] = np.floor(all_vars["trip_time"] / 60)

all_vars["log_driver_pay"] = np.log(all_vars["driver_pay"])
all_vars["log_duration"]   = np.log(all_vars["trip_duration_min"])

# time split
is_train = all_vars["pickup_datetime"] < "2025-01-01"
print(f"train {is_train.sum():,} | test {(~is_train).sum():,}")

# model inputs
OLS_FORMULA = ("log_driver_pay ~ is_JFK + is_LGA + is_EWR "
               "+ C(hour) + C(borough) + log_duration")

GBM_FEATURES = ["hour", "dow", "trip_duration_min", "borough",
                "is_JFK", "is_LGA", "is_EWR"]

X_gbm   = all_vars[GBM_FEATURES]
y_gbm   = all_vars["driver_pay"]          
y_test  = y_gbm[~is_train]                

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train 984,736 | test 951,510


## Baseline

In [21]:
results = {}

def score(name, pred):
    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    results[name] = {"MAE": mae, "RMSE": rmse}
    print(f"{name:<26} MAE ${mae:6.2f}   RMSE ${rmse:6.2f}")

In [22]:
train_df = all_vars[is_train]

group_means = train_df.groupby(["borough", "hour"], observed=True)["driver_pay"].mean()
global_mean = train_df["driver_pay"].mean()

test_keys = list(zip(all_vars.loc[~is_train, "borough"],
                     all_vars.loc[~is_train, "hour"]))
baseline_pred = np.array([group_means.get(k, global_mean) for k in test_keys])

score("Borough-hour mean", baseline_pred)

Borough-hour mean          MAE $ 11.63   RMSE $ 17.47


## Log-linear OLS

In [23]:
import statsmodels.formula.api as smf

ols = smf.ols(OLS_FORMULA, data=all_vars[is_train]).fit(cov_type="HC3")

log_pred = ols.predict(all_vars[~is_train])
sigma2   = ols.resid.var()
ols_pred = np.exp(log_pred) * np.exp(sigma2 / 2)  

score("Log-linear OLS", ols_pred)

Log-linear OLS             MAE $  3.98   RMSE $  8.17


## Gradient boosting

In [28]:
RANDOM_STATE = 42

gbm = HistGradientBoostingRegressor(
    loss="absolute_error",
    max_iter=400,
    learning_rate=0.1,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    categorical_features=["borough"],
    random_state=RANDOM_STATE,
)

gbm.fit(X_gbm[is_train], y_gbm[is_train])
print(f"trees built: {gbm.n_iter_}")

gbm_pred = gbm.predict(X_gbm[~is_train])
score("Gradient boosting", gbm_pred)

trees built: 147
Gradient boosting          MAE $  3.56   RMSE $  7.46


## Comparison

In [29]:
summary = pd.DataFrame(results).T
summary["MAE vs baseline"] = (
    (summary.loc["Borough-hour mean", "MAE"] - summary["MAE"])
    / summary.loc["Borough-hour mean", "MAE"] * 100
).round(1).astype(str) + "%"

summary

,MAE,RMSE,MAE vs baseline
Borough-hour mean,11.633,17.473,0.0%
Log-linear OLS,3.977,8.165,65.8%
Gradient boosting,3.561,7.459,69.4%


## What the model uses

Permutation importance on a test sample.

In [26]:
from sklearn.inspection import permutation_importance

X_test_gbm = X_gbm[~is_train]
sample = X_test_gbm.sample(min(50_000, len(X_test_gbm)), random_state=RANDOM_STATE)

imp = permutation_importance(
    gbm, sample, y_test.loc[sample.index],
    n_repeats=5, random_state=RANDOM_STATE,
    scoring="neg_mean_absolute_error",
)

(pd.Series(imp.importances_mean, index=X_test_gbm.columns)
   .sort_values(ascending=False)
   .to_frame("MAE increase when shuffled ($)"))

,MAE increase when shuffled ($)
trip_duration_min,11.788
hour,0.445
borough,0.214
is_EWR,0.136
is_LGA,0.113
is_JFK,0.097
dow,0.071


## Error by Trip Types

In [30]:
err = pd.DataFrame({
    "actual": y_test.values,
    "pred":   gbm_pred,
    "airport": (X_test_gbm[["is_JFK", "is_LGA", "is_EWR"]].sum(axis=1) > 0).values,
})
err["abs_error"] = (err["actual"] - err["pred"]).abs()

err.groupby("airport")["abs_error"].agg(["mean", "median", "count"])

,mean,median,count
airport,,,
False,3.462,1.409,901361
True,5.338,2.582,50149
